In [2]:
# Create monthly performance data for each salesperson and region
import pandas as pd
sales = pd.DataFrame({
    "salesperson": ["A", "A", "B", "B", "C", "C", "C"],
    "region": ["North", "North", "South", "South", "North", "North", "North"],
    "month": ["Jan", "Feb", "Jan", "Feb", "Jan", "Feb", "Mar"],
    "revenue": [1200, 1500, 900, 1100, 1800, 1600, 1700],
    "orders": [12, 15, 9, 10, 18, 16, 17],
})
sales.index.name="index"
sales

,salesperson,region,month,revenue,orders
index,,,,,
0,A,North,Jan,1200,12
1,A,North,Feb,1500,15
2,B,South,Jan,900,9
3,B,South,Feb,1100,10
4,C,North,Jan,1800,18
5,C,North,Feb,1600,16
6,C,North,Mar,1700,17


In [12]:
# Aggregate total revenue at salesperson level
total_revenue_per_salesperson=sales.groupby("salesperson",as_index=False)["revenue"].sum().rename(columns={"revenue":"total_revenue"})
total_revenue_per_salesperson

,salesperson,total_revenue
0,A,2700
1,B,2000
2,C,5100


In [16]:
# Calculate the average number of orders per salesperson
avg_order_per_salesperson=sales.groupby("salesperson",as_index=False)["orders"].mean().rename(columns={"orders":"avg_orders"})
avg_order_per_salesperson

,salesperson,avg_orders
0,A,13.5
1,B,9.5
2,C,17.0


In [21]:
# Aggregate revenue by region and month
monthly_revenue_per_region=sales.groupby(["region","month"],as_index=False)["revenue"].sum()
monthly_revenue_per_region

,region,month,revenue
0,North,Feb,3100
1,North,Jan,3000
2,North,Mar,1700
3,South,Feb,1100
4,South,Jan,900


In [28]:
# Select the top-revenue salesperson within each region
salesperson_revenue=sales.groupby(["region","salesperson"],as_index=False)["revenue"].sum()
top_salespersons=salesperson_revenue.loc[
    salesperson_revenue.groupby("region")["revenue"].idxmax()
]
top_salespersons

,region,salesperson,revenue
1,North,C,5100
2,South,B,2000


In [8]:
# Rank salespeople within each region by total revenue
salesperson_revenue=(
    sales.groupby(["region","salesperson"],as_index=False)
    .agg(total_revenue=("revenue","sum"))
)
salesperson_revenue["rank"]=(
    salesperson_revenue.groupby("region")["total_revenue"]
    .rank(method="dense",ascending=False)
    .astype(int)
)
salesperson_revenue

,region,salesperson,total_revenue,rank
0,North,A,2700,2
1,North,C,5100,1
2,South,B,2000,1


### Why `groupby().agg()` and `groupby().transform()` Are Not Interchangeable
| Feature | `groupby().agg()` | `groupby().transform()` |
|----------|----------|----------|
| Purpose | Summarize groups | Apply group statistics to each row |
| Output Shape | Reduced | Same as original DataFrame |
| Number of Rows | Usually fewer rows | Same number of rows as input |
| Result Level | Group-level | Row-level |
| Used For | Totals, averages, counts, summaries | Ranking, normalization, percentages, group metrics per row |
| Returns | One value per group (or aggregation) | One value per original row |
| Can Replace the Other? | No | No |

### Important notes

- `groupby().agg()` reduces data to one or more rows per group, while `transform()` preserves the original row count.
- `as_index=False` keeps grouping columns as ordinary columns instead of putting them in the index.
- `idxmax()` returns the first matching row when a group has tied maximum revenue; define a tie-breaker when that matters.
- `rank(method="dense")` gives tied values the same rank without gaps.
- Missing grouping keys are excluded by default; use `dropna=False` when missing groups must be included.
- Confirm that revenue and order fields use consistent units before comparing regions or salespeople.
